# OUARTI — Reprise PPO au checkpoint fixe

**Une seule cellule de reglages, puis « Tout executer ».** Les cellules suivantes definissent les outils ; seule la cellule « Executer la sequence » utilise les interrupteurs. Aucune autre cellule ne les remet a False ou a True.

- Premier lancement apres redemarrage du jeu : immobiliser la voiture sur une portion droite, puis mettre les deux interrupteurs `ACTIVER_SAUVEGARDE_CHECKPOINT` et `ACTIVER_ENTRAINEMENT_REEL` a `True`. La sequence cree le checkpoint avant PPO.
- Checkpoint deja disponible : laisser la sauvegarde a `False` et activer seulement l'entrainement. Avec la sauvegarde a `True`, un checkpoint deja valide est aussi conserve.
- Les deux a `False` : charger/verifier les definitions sans ouvrir de connexion au jeu.

`REPRENDRE_MODELE_EXISTANT=True` conserve le modele et le compteur. Les 4096 actions s'ajoutent aux actions deja apprises. Chaque reprise sauvegarde une copie du modele precedent.

Le checkpoint CSP reste en memoire du jeu : il disparait au redemarrage du jeu. Le modele PPO reste sur disque. Utiliser une seule session locale et un seul notebook de conduite a la fois. Pour arreter l'entrainement, utiliser **Interrompre le noyau** ; modifier un booleen pendant un calcul ne constitue pas un arret d'urgence.

Les episodes sont tronques a la fin des lots PPO, avec frein pendant le calcul puis restauration et observation fraiche apres calcul. `coupure PPO=True` dans le tableau final indique cette troncature, sans penalite d'echec supplementaire.

Ce notebook exige un modele existant et ses metadonnees compatibles.

## Stages et retours arriere
Configuration preparee pour le stage 2 (30 m), 4096 actions. Le modele stage1 reste distinct. Une archive verifiee est conservee avant remplacement du modele principal et en fin de session. Les fichiers sous `models/versions/` sont reserves a la lecture : restaurer via `05_Gerer_Stages_Agents.ipynb` sous un nouveau nom. Le modele source n'est utilise que si le modele cible n'existe pas. Le compteur total est herite ; `stage_timesteps` mesure les actions apprises dans le nouveau stage.


In [ ]:
# REGLAGES — modifier UNIQUEMENT cette cellule, puis cliquer sur Tout executer.
# False/False : charge le code et affiche les reglages, sans connexion au jeu.
ACTIVER_SAUVEGARDE_CHECKPOINT = False  # True : creer le checkpoint s'il manque (voiture immobile)
ACTIVER_ENTRAINEMENT_REEL = False     # True : lancer PPO APRES verification/sauvegarde
REPRENDRE_MODELE_EXISTANT = True      # conserver les poids et le compteur
EXIGER_MODELE_EXISTANT = True       # True dans le notebook 03

TRACK_ATTENDU = 'ks_barcelona-layout_gp'
VOITURE_ATTENDUE = 'abarth500'
MODELE_NOM = 'ppo_vision_checkpoint_v3_stage2'
# Source utilisee seulement si le modele cible n'existe pas.
MODELE_SOURCE_TRANSFERT = 'ppo_vision_checkpoint_v3_stage1'
PENALITE_SORTIE_PISTE = 100.0        # montant positif, SOUSTRAIT lors d une sortie
ETAPE_APPRENTISSAGE = 2              # 30 m
DIRECTION_PROGRESSION = 1
DUREE_EPISODE_S = 25
INTERVALLE_ACTION_S = .12
TOTAL_TIMESTEPS = 4096              # premier essai du stage 2

print('Reglages : sauvegarde si absente =', ACTIVER_SAUVEGARDE_CHECKPOINT,
      '| entrainement =', ACTIVER_ENTRAINEMENT_REEL,
      '| reprise =', REPRENDRE_MODELE_EXISTANT,
      '| actions =', TOTAL_TIMESTEPS)


## Definitions — execution dans l ordre par « Tout executer »

In [ ]:
from pathlib import Path
import sys
CWD=Path.cwd().resolve()
REPO=next((p for p in [CWD,*CWD.parents] if (p/'assetto_corsa_gym'/'assetto_corsa_gym'/'AssettoCorsaPlugin').is_dir()),None)
if REPO is None:
    raise FileNotFoundError('Ouvrir la racine IA_projet_Ouarti_ds_github ou un sous-dossier dans VS Code.')
PROJECT=REPO/'Code'
GYM_ROOT=REPO/'assetto_corsa_gym'/'assetto_corsa_gym'
assert PROJECT.is_dir(), 'Dossier Code introuvable'
if sys.version_info[:2] != (3,9):
    raise RuntimeError('Selectionner le kernel Python 3.9.13 dans VS Code. Actuel : '+sys.version)
for name in ['datasets','models','reports']:
    (PROJECT/name).mkdir(parents=True, exist_ok=True)
print('Depot :',REPO,'\nCode :',PROJECT,'\nPython :',sys.executable)

In [ ]:
import socket, json, ast, time
HOST='127.0.0.1'; PORT_UDP=2345; PORT_TCP=2347

def management(command,timeout=4):
    if command not in ('get_config','get_static_info'):
        raise ValueError('Cette fonction attend une commande qui retourne des donnees.')
    with socket.create_connection((HOST,PORT_TCP),timeout=timeout) as s:
        s.settimeout(timeout)
        s.sendall(command.encode('ascii'))
        data=b''
        deadline=time.monotonic()+timeout
        while time.monotonic()<deadline and len(data)<1024*1024:
            try: chunk=s.recv(65536)
            except socket.timeout: break
            if not chunk: break
            data+=chunk
            try: return json.loads(data)
            except (ValueError,UnicodeError): pass
            try: return ast.literal_eval(data.decode('utf8'))
            except (ValueError,SyntaxError,UnicodeError): pass
    raise RuntimeError('Reponse management absente/incomplete pour '+command+': '+repr(data[:150]))

def request_reset():
    # Le protocole sensors_par NE renvoie AUCUNE reponse a la commande reset.
    with socket.create_connection((HOST,PORT_TCP),timeout=4) as s:
        s.settimeout(4)
        s.sendall(b'reset')
        # Fermer la connexion : le serveur traite la commande sans ACK.

def open_telemetry(timeout=6):
    sock=socket.socket(socket.AF_INET,socket.SOCK_DGRAM)
    sock.settimeout(.30)
    sock.connect((HOST,PORT_UDP))
    end=time.monotonic()+timeout
    try:
        while time.monotonic()<end:
            sock.send(b'connect')
            try: data=sock.recv(2**18)
            except socket.timeout: continue
            if data==b'identified': return sock
        raise TimeoutError('Le plugin UDP 2345 ne repond pas : lancer une session de conduite.')
    except Exception:
        sock.close()
        raise

def next_packet(sock,timeout=3,after_steps=None):
    deadline=time.monotonic()+timeout
    while time.monotonic()<deadline:
        try: data=sock.recv(2**18)
        except socket.timeout: continue
        if data==b'disconnect': raise ConnectionError('Connexion telemetrie interrompue')
        if data==b'identified': continue
        try: packet=json.loads(data)
        except (ValueError,UnicodeError): continue
        if not isinstance(packet,dict): continue
        if after_steps is not None and int(packet.get('steps',-1))<=int(after_steps): continue
        packet['_received_unix']=time.time()
        return packet
    raise TimeoutError('Aucun nouvel echantillon telemetrie dans le delai imparti.')

def close_telemetry(sock):
    if sock is None: return
    try: sock.send(b'disconnect')
    except OSError: pass
    sock.close()

In [ ]:
import sys, time, numpy as np, cv2
PLUGIN_PATH=GYM_ROOT/'AssettoCorsaPlugin'/'plugins'/'sensors_par'
if not PLUGIN_PATH.is_dir():
    raise FileNotFoundError('Plugin Python source manquant : '+str(PLUGIN_PATH))
if str(PLUGIN_PATH) not in sys.path:
    sys.path.insert(0,str(PLUGIN_PATH))
from dual_buffer import GrabberSharedMemoryDualBuffer

class CameraReader:
    def __init__(self,config,age_limit=.55):
        if not config.get('screen_capture_enable'):
            raise RuntimeError('screen_capture_enable=False dans le plugin installe.')
        self.cam=GrabberSharedMemoryDualBuffer(int(config['final_image_width']),
                      int(config['final_image_height']),str(config['color_mode']))
        self.age_limit=float(age_limit)
        self.last_seq=-1
    def read(self,timeout=2,require_new=True):
        deadline=time.monotonic()+float(timeout)
        last_err='Aucune image stable et fraiche'
        while time.monotonic()<deadline:
            try:
                seq,stamp,arr=self.cam.get_timed_image()
                age=time.time()-float(stamp)
                if age<-.15 or age>self.age_limit:
                    last_err='Image perimee ou horloge incoherente (age %.3f s)'%age
                elif require_new and int(seq)<=self.last_seq:
                    last_err='Image non actualisee (seq %s)'%seq
                else:
                    self.last_seq=int(seq)
                    return int(seq),float(stamp),arr
            except (RuntimeError,ValueError) as e:
                last_err=str(e)
            time.sleep(.015)
        raise TimeoutError('Capture indisponible : '+last_err)

def frame_to_image84(frame):
    if frame.ndim==3:
        # La capture couleur du plugin est en BGR.
        gray=cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
    elif frame.ndim==2:
        gray=frame
    else:
        raise ValueError('Format camera invalide : '+str(frame.shape))
    gray=cv2.resize(gray,(84,84),interpolation=cv2.INTER_AREA)
    return np.ascontiguousarray(gray[np.newaxis,:,:],dtype=np.uint8)

In [ ]:
import numpy as np, pandas as pd
from scipy.spatial import cKDTree

class TrackGeometry:
    def __init__(self,path):
        df=pd.read_csv(path)
        required=['left_border_x','left_border_y','right_border_x','right_border_y']
        if not set(required).issubset(df.columns):
            raise ValueError('CSV geometrie non compatible, colonnes attendues: '+str(required))
        left=df[['left_border_x','left_border_y']].to_numpy(dtype=np.float64)
        right=df[['right_border_x','right_border_y']].to_numpy(dtype=np.float64)
        self.center=(left+right)*.5
        self.width=np.linalg.norm(left-right,axis=1)
        self.delta=np.roll(self.center,-1,axis=0)-self.center
        self.segment=np.linalg.norm(self.delta,axis=1)
        if (self.segment>.001).sum()<10:
            raise ValueError('CSV de circuit invalide: pas assez de points')
        self.arc=np.concatenate(([0.],np.cumsum(self.segment)))
        self.length=float(self.arc[-1])
        self.tree=cKDTree(self.center)
    def project(self,raw):
        xy=np.asarray([float(raw['world_position_x']),float(raw['world_position_y'])],dtype=np.float64)
        _,j=self.tree.query(xy)
        j=int(j)
        forward=self.delta[j]/max(float(self.segment[j]),1e-8)
        signed=float(forward[0]*(xy[1]-self.center[j,1])-forward[1]*(xy[0]-self.center[j,0]))
        heading=float(np.arctan2(forward[1],forward[0]))
        return float(self.arc[j]),signed,float(self.width[j]),heading

def numerical_state(raw,track):
    arc,offset,width,heading=track.project(raw)
    diff=(float(raw.get('yaw',0))-heading+np.pi)%(2*np.pi)-np.pi
    vec=[float(raw.get('speed',0))/100.,float(raw.get('accelX',0))/3.,
         float(raw.get('accelY',0))/3.,float(raw.get('angular_velocity_y',0))/4.,
         float(np.clip(offset/max(width/2.,1.),-3,3)),np.sin(diff),np.cos(diff),
         float(raw.get('accStatus',0)),float(raw.get('brakeStatus',0))]
    return np.clip(np.asarray(vec,dtype=np.float32),-5,5),arc,offset,width

def find_track_file(track_name):
    # Les fichiers .pkl sont des grilles d'occupation; ici la recompense
    # utilise les CSV bord gauche/bord droit du depot assetto_corsa_gym.
    name=str(track_name)
    path=GYM_ROOT/'AssettoCorsaConfigs'/'tracks'/(name+'.csv')
    if not path.is_file():
        raise FileNotFoundError('CSV de geometrie introuvable : '+str(path))
    return path

In [ ]:
import ctypes, threading, time, numpy as np
from pathlib import Path

class VJoySafe:
    """Pilote local Windows deja valide dans 04_Test_Volant_vJoy/05_Test_Pedales."""
    X=0x30; Y=0x31; Z=0x32
    def __init__(self,device_id=1,watchdog_sec=.7):
        dll_path=Path(r'C:\Program Files\vJoy\x64\vJoyInterface.dll')
        if not dll_path.exists(): raise FileNotFoundError(str(dll_path))
        self.dll=ctypes.CDLL(str(dll_path)); self.device_id=int(device_id)
        self.dll.GetVJDStatus.argtypes=[ctypes.c_uint]
        self.dll.GetVJDStatus.restype=ctypes.c_int
        self.dll.AcquireVJD.argtypes=[ctypes.c_uint]
        self.dll.AcquireVJD.restype=ctypes.c_int
        self.dll.SetAxis.argtypes=[ctypes.c_long,ctypes.c_uint,ctypes.c_uint]
        self.dll.SetAxis.restype=ctypes.c_int
        self.dll.RelinquishVJD.argtypes=[ctypes.c_uint]
        self.dll.RelinquishVJD.restype=None
        status=self.dll.GetVJDStatus(self.device_id)
        if status!=1: raise RuntimeError('vJoy peripherique %d indisponible, statut=%d'%(self.device_id,status))
        if not self.dll.AcquireVJD(self.device_id): raise RuntimeError('Echec AcquireVJD')
        self.lock=threading.Lock(); self.active=True
        self.last_action=time.monotonic(); self.watchdog_sec=float(watchdog_sec)
        self.last_command={'steer_cmd':0.0, 'throttle_cmd':0.0, 'brake_cmd':1.0}
        self._set_axes(16384,0,32767)  # frein de securite lors de la prise de controle
        self.thread=threading.Thread(target=self._watchdog,daemon=True)
        self.thread.start()
    def _set_axes(self,x,y,z):
        with self.lock:
            for axis,value in ((self.X,x),(self.Y,y),(self.Z,z)):
                if not self.dll.SetAxis(int(value),self.device_id,axis):
                    raise RuntimeError('Echec SetAxis vJoy sur '+hex(axis))
    def action(self,a):
        a=np.asarray(a,dtype=np.float32).reshape(-1)
        if a.shape!=(3,) or not np.isfinite(a).all(): raise ValueError('Action PPO invalide')
        steer=float(np.clip(a[0],-1.,1.))
        gas=float(np.clip(a[1],0.,1.))
        brake=float(np.clip(a[2],0.,1.))
        # Politique PPO au depart aleatoire : accelerateur et frein peuvent etre positifs.
        # Une faible demande de frein ne doit plus annuler un fort accelerateur.
        # Commandes longitudinales mutuellement exclusives : retenir la plus forte.
        if gas >= brake:
            brake = 0.0
        else:
            gas = 0.0
        self._set_axes(round((steer+1)*16383.5),round(gas*32767),round(brake*32767))
        # Ces valeurs sont les commandes *effectivement envoyees* a vJoy.
        self.last_command = {'steer_cmd':steer, 'throttle_cmd':gas, 'brake_cmd':brake}
        self.last_action=time.monotonic()
    def hold(self):
        if self.active:
            self._set_axes(16384,0,32767)
            self.last_action=time.monotonic()
    def _watchdog(self):
        while self.active:
            if time.monotonic()-self.last_action>self.watchdog_sec:
                try: self.hold()
                except Exception: pass
            time.sleep(.06)
    def close(self):
        if not self.active: return
        try:self.hold()
        finally:
            self.active=False
            self.thread.join(timeout=.3)
            self.dll.RelinquishVJD(self.device_id)

In [ ]:
# Cellule de pont entre Jupyter et l'application CSP Lua OUARTICheckpoint.
import json, os, time, uuid, math
from pathlib import Path

BRIDGE_DIR = Path.home() / 'Documents' / 'Assetto Corsa' / 'OUARTI_checkpoint_v3'
ANCHOR_FILE = BRIDGE_DIR / 'anchor.json'

class CheckpointBridge:
    def __init__(self, directory=BRIDGE_DIR):
        self.directory = Path(directory)
        self.command = self.directory / 'command.txt'
        self.response = self.directory / 'status.txt'

    def request(self, command, timeout=8.0):
        if command not in ('PING', 'STATUS', 'SAVE', 'LOAD', 'CAPABILITIES'):
            raise ValueError('Commande CSP inconnue: ' + command)
        if not self.directory.is_dir():
            raise RuntimeError('Application CSP OUARTICheckpoint absente : ' + str(self.directory))
        token = uuid.uuid4().hex
        temporary = self.directory / ('command_' + token + '.tmp')
        temporary.write_text(token + '|' + command, encoding='ascii')
        os.replace(str(temporary), str(self.command))
        deadline = time.monotonic() + float(timeout)
        last = ''
        while time.monotonic() < deadline:
            try:
                last = self.response.read_text(encoding='utf8').strip()
            except (FileNotFoundError, UnicodeError, PermissionError):
                time.sleep(0.05)
                continue
            parts = last.split('|', 2)
            if len(parts) == 3 and parts[0] == token:
                if parts[1] == 'ERROR':
                    raise RuntimeError('CSP checkpoint: ' + parts[2])
                if parts[1] == 'PENDING':
                    time.sleep(0.05)
                    continue
                return parts[1], parts[2]
            time.sleep(0.05)
        raise TimeoutError('Pas de reponse CSP pour ' + command + '. Dernier statut: ' + last[:140])

    def capabilities(self):
        code, detail = self.request('CAPABILITIES')
        if code != 'CAPABILITIES':
            raise RuntimeError('Reponse CAPABILITIES inattendue: ' + code + ' ' + detail)
        fields = {}
        for field in detail.split(';'):
            key, separator, value = field.strip().partition('=')
            if separator:
                fields[key.strip()] = value.strip()
        for key in ('offline', 'full_snapshot', 'position_fallback'):
            if fields.get(key) not in ('true', 'false'):
                raise RuntimeError('Capacite CSP absente ou invalide: ' + key + ': ' + detail)
        return {key: value == 'true' for key, value in fields.items()
                if key in ('offline', 'full_snapshot', 'position_fallback')}

    def assert_supported(self):
        caps = self.capabilities()
        if not caps['offline']:
            raise RuntimeError('Checkpoint reserve a une session locale hors ligne.')
        if not (caps['full_snapshot'] or caps['position_fallback']):
            raise RuntimeError(
                'Checkpoint indisponible: CSP ne fournit ni snapshot complet ni '
                'repositionnement autorise dans cette session. La connexion au pont fonctionne. '
                'Verifier la version CSP dans Content Manager > Settings > Custom Shaders Patch '
                '> About & Updates. Sur le PC examine le 16/11/2026, CSP 0.1.79 est installe '
                'et son SDK ne declare pas saveCarStateAsync/loadCarState. '
                'Apres mise a jour compatible avec sensors_par, relancer le jeu et refaire '
                'PING puis CAPABILITIES. Si les deux capacites restent false, ne pas lancer PPO. '
                'Le fallback position depend aussi de physics.allowed(); '
                'une mise a jour seule ne garantit pas son activation.')
        return caps

    def assert_saved(self):
        code, detail = self.request('STATUS')
        if code != 'SAVED':
            raise RuntimeError('Checkpoint non enregistre dans cette session CSP: ' + detail
                               + '. Executer la cellule de sauvegarde avec ACTIVER_SAUVEGARDE_CHECKPOINT=True, '
                               'voiture immobile, puis remettre False. Refaire SAVE apres chaque redemarrage du jeu.')
        return detail

    def load(self):
        code, detail = self.request('LOAD', timeout=8)
        if code != 'LOAD_REQUESTED':
            raise RuntimeError('CSP n a pas confirme la commande LOAD: ' + code + ' ' + detail)


def telemetry_xy(raw):
    return float(raw['world_position_x']), float(raw['world_position_y'])


def distance_from_anchor(raw, anchor):
    x, y = telemetry_xy(raw)
    xa, ya = float(anchor['x']), float(anchor['y'])
    distance = math.hypot(x - xa, y - ya)
    angle_error = abs((float(raw['yaw']) - float(anchor['yaw']) + math.pi) % (2 * math.pi) - math.pi)
    return distance, angle_error


def make_anchor_from_raw(raw, track, car):
    x, y = telemetry_xy(raw)
    if float(raw.get('speed', 999)) > 0.3:
        raise RuntimeError('Immobiliser la voiture avant sauvegarde (<1 km/h).')
    return {'track': track, 'car': car, 'x': x, 'y': y,
            'yaw': float(raw['yaw']), 'created_at': time.time(),
            'note': 'Checkpoint CSP en memoire, non reutilisable apres redemarrage du jeu.'}


In [ ]:
import os, json, time, numpy as np, gymnasium as gym
from gymnasium import spaces

class LiveAssettoVisionEnv(gym.Env):
    """1 instance = 1 jeu local, 1 connexion UDP, 1 vJoy. Jamais de simulation sans jeu."""
    metadata={'render_modes':[]}
    def __init__(self,expected_track,expected_car,episode_seconds=45,
                 action_interval=.12,progress_sign=1,log_file=None,curriculum_stage=1,offtrack_penalty=100.0):
        super().__init__()
        self.offtrack_penalty=float(offtrack_penalty)
        if not np.isfinite(self.offtrack_penalty) or self.offtrack_penalty <= 0:
            raise ValueError('La penalite de sortie doit etre un nombre positif fini.')
        self.expected_track=str(expected_track)
        self.expected_car=str(expected_car)
        self.curriculum_stage=int(curriculum_stage)
        self.goals={1:10.0,2:30.0,3:100.0,4:None}
        if self.curriculum_stage not in self.goals:
            raise ValueError('Etape du programme: 1, 2, 3 ou 4.')
        self.goal_m=self.goals[self.curriculum_stage]
        self.episode_progress=0.0
        self.rollout_remaining=None
        self.progress_sign=int(progress_sign)
        if self.progress_sign not in (-1,1): raise ValueError('progress_sign doit valoir -1 ou 1')
        self.episode_seconds=float(episode_seconds)
        self.action_interval=float(action_interval)
        self.action_space=spaces.Box(low=np.array([-1,0,0],dtype=np.float32),
                                     high=np.ones(3,dtype=np.float32),dtype=np.float32)
        self.observation_space=spaces.Dict({
            'image':spaces.Box(low=0,high=255,shape=(1,84,84),dtype=np.uint8),
            'state':spaces.Box(low=-5.,high=5.,shape=(9,),dtype=np.float32)})
        config=management('get_config'); sess=management('get_static_info')
        track=sess.get('TrackName','')
        if sess.get('TrackConfiguration'): track+='-'+str(sess['TrackConfiguration'])
        car=str(sess.get('CarName',''))
        if track!=self.expected_track or car!=self.expected_car:
            raise RuntimeError('Mauvaise session : '+track+'/'+car+' attendu '+self.expected_track+'/'+self.expected_car)
        if bool(config.get('vjoy_executed_by_server')):
            raise RuntimeError('Ce mode utilise vJoy local: laisser vjoy_executed_by_server=False.')
        if not config.get('screen_capture_enable'):
            raise RuntimeError('Capture camera desactivee dans le plugin')
        if sess.get('autoShifterOn') is False:
            raise RuntimeError('Activer la boite automatique dans Assetto Corsa avant PPO.')
        self.track=TrackGeometry(find_track_file(track))
        self.camera=CameraReader(config)
        self.sock=None; self.controller=None
        self.raw=None; self.arc=None;self.damage=0.;self.lap=0
        self.t0=0.;self.last_obs=None;self.idle_since=0.;self.episode_index=0;self.steps=0
        self.last_action=np.zeros(3,np.float32)
        self.log_file=Path(log_file) if log_file else None
        self.bridge=CheckpointBridge()
        self.bridge.request('PING', timeout=5)
        self.bridge.assert_supported()
        self.bridge.assert_saved()
        if not ANCHOR_FILE.is_file():
            raise RuntimeError('Point de depart inconnu : utiliser la cellule SAUVEGARDER CHECKPOINT avant PPO.')
        self.anchor=json.loads(ANCHOR_FILE.read_text(encoding='utf8'))
        if self.anchor.get('track')!=track or self.anchor.get('car')!=car:
            raise RuntimeError('Checkpoint sauvegarde pour une autre voiture ou un autre circuit.')
        if self.log_file: self.log_file.parent.mkdir(parents=True,exist_ok=True)
        print('Config controle local: vJoy=local, plugin',config.get('sampling_freq'),'Hz, camera',config['color_mode'])
    def begin_rollout(self, steps):
        self.rollout_remaining = int(steps)
        if self.rollout_remaining <= 0:
            raise ValueError('Nombre de pas du lot invalide')

    def safe_stop(self):
        if self.controller is not None: self.controller.hold()
    def _connect(self):
        if self.controller is None:
            self.controller=VJoySafe(device_id=1)
        if self.sock is None:
            self.sock=open_telemetry()
    def _drain_pending_telemetry(self, max_packets=4096, max_seconds=0.12):
        """Vide les vieux datagrammes UDP sans bloquer ni perdre le timeout initial.

        A appeler apres le reset et avant la lecture de chaque observation :
        pendant time.sleep(), le buffer UDP conserve des mesures deja perimees.
        """
        if self.sock is None:
            return 0
        timeout_before = self.sock.gettimeout()
        removed = 0
        deadline = time.monotonic() + float(max_seconds)
        try:
            self.sock.setblocking(False)
            while removed < max_packets and time.monotonic() < deadline:
                try:
                    self.sock.recv(2**18)
                    removed += 1
                except (BlockingIOError, InterruptedError):
                    break
        finally:
            self.sock.settimeout(timeout_before)
        return removed

    def _observe(self, after_steps=None):
        """Obtient une image et une telemetrie recentes, avec delai controle.

        Si elles ne correspondent pas, on reprend une PAIRE fraiche plutot
        que de conserver la telemetrie d'un essai precedent.
        """
        deadline = time.monotonic() + 4.0
        last_error = 'Aucune paire image/telemetrie valide'
        while time.monotonic() < deadline:
            self._drain_pending_telemetry()
            remaining = deadline - time.monotonic()
            if remaining <= 0:
                break
            try:
                raw = next_packet(
                    self.sock,
                    timeout=min(0.8, remaining),
                    after_steps=after_steps,
                )
                remaining = deadline - time.monotonic()
                if remaining <= 0:
                    break
                seq, stamp, frame = self.camera.read(
                    timeout=min(0.8, remaining),
                )
            except (TimeoutError, RuntimeError, OSError, ConnectionError) as exc:
                last_error = str(exc)
                continue

            packet_stamp = float(raw.get('timestamp_unix', raw['_received_unix']))
            gap = abs(float(stamp) - packet_stamp)
            if gap > 0.40:
                now = time.time()
                last_error = (
                    'Camera/telemetrie non synchronisees: ecart %.3f s '
                    '(age image %.3f s, age telemetrie %.3f s)'
                    % (gap, now - float(stamp), now - packet_stamp)
                )
                continue

            state, arc, lateral, width = numerical_state(raw, self.track)
            obs = {
                'image': frame_to_image84(frame),
                'state': state,
            }
            return raw, obs, arc, lateral, width, seq, gap
        raise TimeoutError('Impossible de synchroniser les capteurs: ' + last_error)
    def reset(self,*,seed=None,options=None):
        super().reset(seed=seed)
        self._connect()
        self.safe_stop()
        self.bridge.assert_saved()
        # Ici on RESTAURE un snapshot CSP, pas un reset local sans retour au depart.
        self.bridge.load()
        time.sleep(0.45)
        self._drain_pending_telemetry()
        last_err='Pas de telemetrie valide apres restauration'
        deadline=time.monotonic()+10.0
        while time.monotonic()<deadline:
            try:
                raw,o,arc,lat,width,seq,gap=self._observe()
                distance,angle=distance_from_anchor(raw,self.anchor)
                speed=float(raw.get('speed',999.))
                if distance>1.5 or angle>0.30 or speed>0.75:
                    last_err=('Checkpoint non restaure: position %.2f m, orientation %.2f rad, '
                              'vitesse %.2f m/s' % (distance,angle,speed))
                    time.sleep(0.08)
                    continue
                break
            except (TimeoutError,RuntimeError,OSError,ConnectionError,ValueError) as e:
                last_err=str(e)
        else:
            self.safe_stop()
            raise RuntimeError('RESET FIXE REFUSE : '+last_err+'. Entrainer = False.')
        self.raw=raw;self.last_obs=o;self.arc=arc
        self.damage=float(raw.get('damage_total',0))
        self.lap=int(raw.get('LapCount',0))
        self.idle_since=time.monotonic()
        self.t0=time.monotonic();self.steps=0;self.episode_index+=1
        self.last_action=np.zeros(3,np.float32)
        self.episode_progress=0.0
        print('Reset fixe confirme: %.2f m / %.2f rad / %.2f m/s' % (distance,angle,speed))
        return o,{'camera_gap_s':gap,'camera_sequence':seq,'track':self.expected_track,
                  'reset_verified':True,'reset_distance_m':distance,'reset_yaw_error_rad':angle,
                  'curriculum_stage':self.curriculum_stage,'goal_distance_m':self.goal_m}
    def step(self,action):
        if self.sock is None or self.raw is None:
            raise RuntimeError('reset() doit etre appele avant step()')
        if self.rollout_remaining is not None:
            self.rollout_remaining -= 1
        rollout_cut = self.rollout_remaining == 0
        action=np.asarray(action,dtype=np.float32)
        try:
            self.controller.action(action)
            time.sleep(self.action_interval)
            raw,o,arc,lateral,width,seq,gap=self._observe(after_steps=self.raw.get('steps'))
            dt=float(raw.get('timestamp_unix',raw['_received_unix']))-float(self.raw.get('timestamp_unix',self.raw['_received_unix']))
            if not .005<dt<1.0: raise RuntimeError('Delta temps telemetrie invalide: %s'%dt)
            move=(arc-self.arc+self.track.length/2)%self.track.length-self.track.length/2
            move*=self.progress_sign
            if abs(move)>max(4.,100.*dt*1.7):
                # Collision, teleportation, mauvaise projection ou horodatage : episode invalide.
                raise RuntimeError('Saut de position non physique (%.1f m sur %.3f s)'%(move,dt))
            tyres=int(raw.get('numberOfTyresOut',0))
            offtrack=bool(tyres>2 or abs(lateral)>width*.5)
            new_damage=float(raw.get('damage_total',0))
            collision=bool(new_damage>self.damage+.02)
            new_lap=int(raw.get('LapCount',0))
            lap_finished=new_lap>self.lap
            now=time.monotonic(); speed=float(raw.get('speed',0))
            if speed>1.:self.idle_since=now
            idle=bool(now-self.idle_since>6.)
            wrong_way=bool(move<-.25)
            # Curriculum sans imitation : progres geometrie, puis distances croissantes.
            self.episode_progress+=float(move)
            milestone=bool(self.goal_m is not None and self.episode_progress>=self.goal_m
                           and not offtrack and not collision)
            if self.curriculum_stage==1:
                reward=float(2.5*np.clip(move,-3.,5.)-.02
                             -.006*abs(float(action[0])-float(self.last_action[0]))
                             -self.offtrack_penalty*offtrack-25.*collision-5.*idle-2.*wrong_way
                             +20.*milestone)
            else:
                reward=float(np.clip(move,-5.,8.)-.02
                             -.015*abs(float(o['state'][4]))
                             -.008*abs(float(action[0])-float(self.last_action[0]))
                             -self.offtrack_penalty*offtrack-35.*collision-8.*idle-2.*wrong_way
                             +30.*milestone+70.*(lap_finished and not offtrack and not collision))
            terminated=bool(offtrack or collision or idle or milestone or lap_finished)
            time_limit = bool(now-self.t0>=self.episode_seconds)
            truncated=bool(time_limit or (rollout_cut and not terminated))
            self.last_obs=o;self.raw=raw;self.arc=arc;self.damage=new_damage;self.lap=new_lap
            self.last_action=action.copy();self.steps+=1
            # Enregistrer les commandes reelles et les retours des pedales.
            # Utile pour distinguer exploration PPO et probleme de configuration du jeu.
            sent=dict(self.controller.last_command)
            def telemetry_number(*keys):
                for key in keys:
                    if key in raw and raw[key] is not None:
                        try: return float(raw[key])
                        except (ValueError,TypeError): pass
                return None
            info={'progress_m':float(move),'speed_ms':speed,'offtrack':offtrack,'collision':collision,
                  'offtrack_penalty':float(self.offtrack_penalty*offtrack),
                  'lap_finished':lap_finished,'idle':idle,'wrong_way':wrong_way,'camera_gap_s':gap,
                  'camera_sequence':seq,'episode_id':self.episode_index,
                  'episode_progress_m':float(self.episode_progress),
                  'milestone_reached':bool(milestone),'goal_distance_m':self.goal_m,
                  'curriculum_stage':self.curriculum_stage,
                  'rollout_cut':bool(rollout_cut and not terminated),
                  'time_limit':time_limit,
                  **sent,
                  'accStatus':telemetry_number('accStatus'),
                  'brakeStatus':telemetry_number('brakeStatus'),
                  'rpm':telemetry_number('rpm','RPM','engineRPM'),
                  'gear':telemetry_number('gear','Gear','currentGear','CurrentGear')}
            if terminated or truncated: self.safe_stop()
            if self.log_file:
                with self.log_file.open('a',encoding='utf8') as f:
                    f.write(json.dumps({'time':time.time(),'reward':reward,'terminated':terminated,
                             'truncated':truncated,**info},ensure_ascii=False)+'\n')
            return o,reward,terminated,truncated,info
        except (OSError,RuntimeError,TimeoutError,ConnectionError,ValueError) as exc:
            self.safe_stop()
            if self.last_obs is None:raise
            return self.last_obs,-40.,False,True,{'sensor_error':str(exc),'episode_id':self.episode_index}
    def close(self):
        self.safe_stop()
        if self.sock is not None:
            close_telemetry(self.sock);self.sock=None
        if self.controller is not None:
            self.controller.close();self.controller=None

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor
import json, time, torch, shutil, uuid

CONTROL_SCHEMA='vjoy_local_xyz_pedales_exclusives_v3'
OBS_SCHEMA='dict_image_1x84x84_state_9'

import hashlib, re


def empreinte_fichier(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(1024*1024),b''):
            digest.update(block)
    return digest.hexdigest()


def prefix_modele(selection):
    path=Path(selection)
    if not path.is_absolute():path=PROJECT/'models'/path
    if path.suffix=='.zip':path=path.with_suffix('')
    return path.resolve()


def fichiers_modele(prefix):
    prefix=Path(prefix)
    return prefix.with_suffix('.zip'),prefix.with_name(prefix.name+'_meta.json')


def lire_modele(selection):
    prefix=prefix_modele(selection);model,metadata=fichiers_modele(prefix)
    if not model.is_file() or not metadata.is_file():
        raise FileNotFoundError('ZIP et metadonnees requis : '+str(prefix))
    versions=(PROJECT/'models'/'versions').resolve()
    if versions in prefix.parents:
        manifest_path=prefix.parent/'manifest.json'
        if not manifest_path.is_file():raise RuntimeError('Archive sans manifeste : '+str(prefix))
        manifest=json.loads(manifest_path.read_text(encoding='utf8'))
        if empreinte_fichier(model)!=manifest.get('zip_sha256') or empreinte_fichier(metadata)!=manifest.get('meta_sha256'):
            raise RuntimeError('Archive alteree, lecture refusee : '+str(prefix))
    return prefix,json.loads(metadata.read_text(encoding='utf8'))


def verifier_destination_modifiable(prefix):
    prefix=Path(prefix).resolve()
    versions=(PROJECT/'models'/'versions').resolve()
    if prefix==versions or versions in prefix.parents:
        raise RuntimeError('Les archives versions sont protegees. Creer un agent de travail sous un nouveau nom.')


def destination_agent(nom):
    if not isinstance(nom,str) or re.fullmatch(r'[A-Za-z0-9_-]+',nom) is None:
        raise ValueError('Nom de modele : lettres, chiffres, tirets et underscores uniquement.')
    prefix=(PROJECT/'models'/nom).resolve()
    verifier_destination_modifiable(prefix)
    return prefix


def archiver_agent(selection,motif='avant_modification'):
    prefix,meta=lire_modele(selection)
    model,metadata=fichiers_modele(prefix)
    hashes={'zip_sha256':empreinte_fichier(model),'meta_sha256':empreinte_fichier(metadata)}
    versions=PROJECT/'models'/'versions'
    # The exact same ZIP/metadata pair only needs one archive.
    for existing in versions.glob('*/*/manifest.json'):
        manifest=json.loads(existing.read_text(encoding='utf8'))
        if all(manifest.get(k)==v for k,v in hashes.items()):
            archived=existing.parent/'agent'
            a,b=fichiers_modele(archived)
            if not a.is_file() or not b.is_file() or empreinte_fichier(a)!=hashes['zip_sha256'] or empreinte_fichier(b)!=hashes['meta_sha256']:
                raise RuntimeError('Archive alteree : '+str(existing.parent))
            return archived
    label=re.sub(r'[^A-Za-z0-9_-]','_',str(meta.get('model',prefix.name)))
    folder=versions/label/(time.strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8])
    folder.mkdir(parents=True,exist_ok=False)
    archived=folder/'agent';a,b=fichiers_modele(archived)
    shutil.copy2(model,a);shutil.copy2(metadata,b)
    if empreinte_fichier(a)!=hashes['zip_sha256'] or empreinte_fichier(b)!=hashes['meta_sha256']:
        raise RuntimeError('Copie archive non conforme : '+str(folder))
    manifest={'version_format':1,'source':str(prefix),'motif':motif,'created_at':time.time(),
        'model':meta.get('model',prefix.name),'curriculum_stage':meta.get('curriculum_stage'),
        'goal_m':meta.get('goal_m'),'total_timesteps':meta.get('total_timesteps'),**hashes}
    (folder/'manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf8')
    return archived


def creer_agent_depuis(source,nom_cible,stage_cible=None):
    """Fork a verified archived model. Never overwrite an existing target."""
    original,meta=lire_modele(source)
    target=destination_agent(nom_cible)
    out_zip,out_meta=fichiers_modele(target)
    if out_zip.exists() or out_meta.exists():
        raise FileExistsError('Destination deja presente, choisir un nouveau nom : '+str(target))
    if meta.get('control')!='vjoy_local_xyz_pedales_exclusives_v3' or meta.get('observation')!='dict_image_1x84x84_state_9' or meta.get('reset')!='csp_snapshot_verified':
        raise RuntimeError('Source incompatible avec le pilote V3.')
    current=meta.get('curriculum_stage')
    if type(current) is not int or current not in (1,2,3,4):
        raise RuntimeError('Etape source inconnue.')
    stage=current if stage_cible is None else stage_cible
    if type(stage) is not int or stage not in (current,current+1) or stage not in (1,2,3,4):
        raise ValueError('Autorise : copie de la meme etape ou transfert vers la suivante uniquement.')
    archived=archiver_agent(original,'source_transfert' if stage!=current else 'source_restauration')
    source_zip,source_meta=fichiers_modele(archived)
    # Build a new metadata tree; original metadata stays byte-for-byte intact.
    updated={**meta,'model':nom_cible,'curriculum_stage':stage,
        'goal_m':{1:10.,2:30.,3:100.,4:None}[stage],
        'parent_version':str(archived.relative_to((PROJECT/'models').resolve())),
        'parent_stage':current,'parent_total_timesteps':int(meta['total_timesteps']),
        'initialization':'stage_transfer' if stage!=current else 'restored_copy',
        'created_at':time.time(),'saved_at':time.time()}
    for key in ('run_id','log_dir'):updated.pop(key,None)
    if stage!=current:
        updated['stage_start_timesteps']=int(meta['total_timesteps'])
        updated['stage_timesteps']=0
    # Exclusive creation also prevents accidental replacement on repeated execution.
    out_zip.parent.mkdir(parents=True,exist_ok=True)
    with source_zip.open('rb') as src,out_zip.open('xb') as dst:
        shutil.copyfileobj(src,dst)
    with out_meta.open('x',encoding='utf8') as dst:
        json.dump(updated,dst,indent=2,ensure_ascii=False)
    if empreinte_fichier(out_zip)!=empreinte_fichier(source_zip):
        raise RuntimeError('Le nouvel agent ne correspond pas aux poids sources.')
    # Also freeze the initial state of the child, before its first action.
    initial=archiver_agent(target,'initialisation_agent')
    print('Agent cree :',out_zip,'| stage',stage,'| compteur herite',updated['total_timesteps'])
    print('Source conservee :',archived)
    print('Version initiale conservee :',initial)
    return target


def inventorier_versions():
    rows=[]
    for path in sorted((PROJECT/'models'/'versions').glob('*/*/manifest.json')):
        meta=json.loads(path.read_text(encoding='utf8'))
        rows.append({'modele':meta.get('model'),'stage':meta.get('curriculum_stage'),
            'compteur':meta.get('total_timesteps'),'motif':meta.get('motif'),
            'source_selectionnable':str((path.parent/'agent').relative_to(PROJECT/'models'))})
    return rows



def validate_resume_metadata(meta):
    expected = {'track':TRACK_ATTENDU, 'car':VOITURE_ATTENDUE,
                'control':CONTROL_SCHEMA, 'observation':OBS_SCHEMA,
                'reset':'csp_snapshot_verified',
                'progress_sign':DIRECTION_PROGRESSION,
                'curriculum_stage':ETAPE_APPRENTISSAGE}
    mismatch = {key:(meta.get(key), value) for key,value in expected.items()
                if meta.get(key) != value}
    if mismatch:
        raise RuntimeError('Modele incompatible avec cette session/configuration: ' + str(mismatch))


def save_training(agent, prefix, meta):
    # Remplacement atomique de chaque fichier : eviter les ZIP partiellement ecrits.
    prefix=Path(prefix)
    verifier_destination_modifiable(prefix)
    previous_zip,previous_meta=fichiers_modele(prefix)
    if previous_zip.exists() or previous_meta.exists():
        archiver_agent(prefix,'avant_remplacement')
    prefix.parent.mkdir(parents=True,exist_ok=True)
    temporary=prefix.with_name(prefix.name+'_writing_'+uuid.uuid4().hex)
    updated={**meta,'total_timesteps':int(agent.num_timesteps),'saved_at':time.time()}
    if 'stage_start_timesteps' in updated:
        updated['stage_timesteps']=int(agent.num_timesteps)-int(updated['stage_start_timesteps'])
    temporary_meta=temporary.with_name(temporary.name+'_meta.json')
    try:
        agent.save(str(temporary))
        temporary_meta.write_text(json.dumps(updated,indent=2),encoding='utf8')
        os.replace(str(temporary.with_suffix('.zip')),str(prefix.with_suffix('.zip')))
        os.replace(str(temporary_meta),str(prefix.with_name(prefix.name+'_meta.json')))
    finally:
        for leftover in (temporary.with_suffix('.zip'),temporary_meta):
            if leftover.exists(): leftover.unlink()
    if prefix.parent.resolve()==(PROJECT/'models').resolve():
        archiver_agent(prefix,'sauvegarde_agent')
    return updated


class PauseDuringUpdates(BaseCallback):
    """Coupure explicite du lot, frein pendant le calcul, puis reset frais."""
    def __init__(self,core,prefix,meta,save_every=1024):
        super().__init__()
        self.core=core; self.prefix=prefix; self.meta=meta
        self.save_every=int(save_every)
        self.paused=False; self.last_saved=0

    def _on_training_start(self):
        if self.training_env.num_envs != 1:
            raise RuntimeError('Une seule instance Assetto Corsa est supportee.')
        self.last_saved=int(self.model.num_timesteps)

    def _on_rollout_start(self):
        # Sauvegarder avant le reset frais : le temps disque reste hors episode.
        if self.model.num_timesteps-self.last_saved >= self.save_every:
            dest=self.prefix.parent/'checkpoints'/(self.prefix.name+'_'+str(self.model.num_timesteps)+'_steps_'+self.meta['run_id'])
            save_training(self.model,dest,self.meta)
            self.last_saved=int(self.model.num_timesteps)
        if self.paused:
            # VecEnv a deja reset apres la coupure. Refaire le reset APRES le
            # calcul, car le jeu a continue sous frein pendant la mise a jour.
            self.model._last_obs=self.training_env.reset()
            self.model._last_episode_starts=np.ones((1,),dtype=bool)
            self.paused=False
        self.core.begin_rollout(self.model.n_steps)

    def _on_step(self):
        return True

    def _on_rollout_end(self):
        self.core.safe_stop()
        self.paused=True

    def _on_training_end(self):
        self.core.safe_stop()




In [ ]:
def sauvegarder_checkpoint_si_absent(bridge, track, car):
    """Creation autorisee par le reglage; ne remplace pas un depart deja valide."""
    code, detail=bridge.request('STATUS')
    if code == 'SAVED' and ANCHOR_FILE.is_file():
        anchor=json.loads(ANCHOR_FILE.read_text(encoding='utf8'))
        if (anchor.get('track'),anchor.get('car')) == (track,car):
            print('Checkpoint deja enregistre : conservation du depart existant.')
            return
        raise RuntimeError('Le fichier anchor.json correspond a une autre session. '
                           'Recreer volontairement le checkpoint dans une nouvelle session du jeu.')
    sock=open_telemetry()
    try:
        sample=next_packet(sock,timeout=3)
        make_anchor_from_raw(sample,track,car)  # exige une voiture immobile
        print('Sauvegarde du checkpoint CSP...')
        code, detail=bridge.request('SAVE',timeout=15)
        if code != 'SAVED':
            raise RuntimeError('Sauvegarde CSP non confirmee : '+code+' '+detail)
        sample=next_packet(sock,timeout=3)
        anchor=make_anchor_from_raw(sample,track,car)
        ANCHOR_FILE.write_text(json.dumps(anchor,indent=2),encoding='utf8')
        bridge.assert_saved()
        print('Checkpoint enregistre :',anchor)
    finally:
        close_telemetry(sock)


def executer_ouarti():
    """Un seul point d'entree : diagnostic -> sauvegarde eventuelle -> PPO."""
    switches=(ACTIVER_SAUVEGARDE_CHECKPOINT,ACTIVER_ENTRAINEMENT_REEL,
              REPRENDRE_MODELE_EXISTANT,EXIGER_MODELE_EXISTANT)
    if any(type(value) is not bool for value in switches):
        raise ValueError('Les interrupteurs doivent etre True ou False, sans guillemets.')
    if not (ACTIVER_SAUVEGARDE_CHECKPOINT or ACTIVER_ENTRAINEMENT_REEL):
        print('Mode desactive : aucune connexion au jeu, aucune commande.')
        print('Modifier la cellule REGLAGES puis Tout executer.')
        return
    if ETAPE_APPRENTISSAGE not in (1,2,3,4) or DIRECTION_PROGRESSION not in (-1,1):
        raise ValueError('Etape ou sens de progression invalide.')
    if int(TOTAL_TIMESTEPS) <= 0 or DUREE_EPISODE_S <= 0 or INTERVALLE_ACTION_S <= 0:
        raise ValueError('Nombre de pas et durees doivent etre positifs.')
    if not np.isfinite(PENALITE_SORTIE_PISTE) or PENALITE_SORTIE_PISTE <= 0:
        raise ValueError('PENALITE_SORTIE_PISTE doit etre positive et finie.')
    print('Penalite de sortie de piste : -', PENALITE_SORTIE_PISTE)
    print('1/3 Verification du pont et de la session...')
    bridge=CheckpointBridge()
    print('Application :',bridge.request('PING'))
    print('Capacites :',bridge.assert_supported())
    session=management('get_static_info')
    track=session.get('TrackName','')
    if session.get('TrackConfiguration'):
        track+='-'+str(session['TrackConfiguration'])
    car=str(session.get('CarName',''))
    if (track,car)!=(TRACK_ATTENDU,VOITURE_ATTENDUE):
        raise RuntimeError('Session inattendue : '+track+'/'+car)
    print('2/3 Checkpoint...')
    if ACTIVER_SAUVEGARDE_CHECKPOINT:
        sauvegarder_checkpoint_si_absent(bridge,track,car)
    elif ACTIVER_ENTRAINEMENT_REEL:
        code,detail=bridge.request('STATUS')
        if code != 'SAVED' or not ANCHOR_FILE.is_file():
            raise RuntimeError('Checkpoint absent ou reference manquante. Immobiliser la voiture, '
                               'mettre ACTIVER_SAUVEGARDE_CHECKPOINT=True dans REGLAGES, '
                               'puis Tout executer. La sequence sauvegardera avant PPO.')
    if ACTIVER_ENTRAINEMENT_REEL:
        bridge.assert_saved()
        print('3/3 Entrainement (reprise si modele present)...')
        entrainer_ppo()
    else:
        print('3/3 Entrainement desactive : checkpoint pret, aucune conduite lancee.')


In [ ]:
def entrainer_ppo():
    core=None; agent=None
    prefix=destination_agent(MODELE_NOM)
    meta_file=prefix.with_name(prefix.name+'_meta.json')
    model_file=prefix.with_suffix('.zip')
    run_id=time.strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8]
    log_dir=PROJECT/'reports'/('ppo_checkpoint_v3_stage'+str(ETAPE_APPRENTISSAGE))/run_id
    log_dir.mkdir(parents=True,exist_ok=True)
    if not model_file.exists() and MODELE_SOURCE_TRANSFERT and not EXIGER_MODELE_EXISTANT:
        source_prefix,source_meta=lire_modele(MODELE_SOURCE_TRANSFERT)
        if ETAPE_APPRENTISSAGE!=source_meta.get('curriculum_stage',0)+1:
            raise ValueError('Le transfert doit aller vers le stage suivant. Pour revenir, utiliser le notebook de versions.')
        expected_source={**source_meta,'curriculum_stage':ETAPE_APPRENTISSAGE}
        validate_resume_metadata(expected_source)
        creer_agent_depuis(source_prefix,MODELE_NOM,ETAPE_APPRENTISSAGE)
    resume=bool(REPRENDRE_MODELE_EXISTANT and model_file.is_file())
    if EXIGER_MODELE_EXISTANT and not resume:
        raise FileNotFoundError('Modele requis pour reprise: '+str(model_file))
    if model_file.exists() and not REPRENDRE_MODELE_EXISTANT:
        raise FileExistsError('Choisir un nouveau MODELE_NOM pour repartir de zero : '+str(model_file))
    previous_meta={}
    if resume:
        previous_meta=json.loads(meta_file.read_text(encoding='utf8'))
        validate_resume_metadata(previous_meta)
        version=archiver_agent(prefix,'avant_reprise')
        print('Version de retour conservee :',version)
    meta={'model':MODELE_NOM,'track':TRACK_ATTENDU,'car':VOITURE_ATTENDUE,
          'progress_sign':DIRECTION_PROGRESSION,'control':CONTROL_SCHEMA,
          'observation':OBS_SCHEMA,'image_color':'gray_from_plugin',
          'curriculum_stage':ETAPE_APPRENTISSAGE,'reset':'csp_snapshot_verified',
          'goal_m':{1:10.,2:30.,3:100.,4:None}[ETAPE_APPRENTISSAGE],
          'episode_boundary':'rollout_truncated_then_reset_after_update',
          'run_id':run_id,'log_dir':str(log_dir),
          'episode_seconds':DUREE_EPISODE_S,'action_interval':INTERVALLE_ACTION_S,
          'reward_version':'offtrack_penalty_v2', 'offtrack_penalty':PENALITE_SORTIE_PISTE}
    for key in ('parent_version','parent_stage','parent_total_timesteps','initialization','created_at','stage_start_timesteps','stage_timesteps'):
        if key in previous_meta:meta[key]=previous_meta[key]
    try:
        core=LiveAssettoVisionEnv(TRACK_ATTENDU,VOITURE_ATTENDUE,
             episode_seconds=DUREE_EPISODE_S,action_interval=INTERVALLE_ACTION_S,
             progress_sign=DIRECTION_PROGRESSION,log_file=log_dir/'ppo_steps.jsonl',
             curriculum_stage=ETAPE_APPRENTISSAGE,offtrack_penalty=PENALITE_SORTIE_PISTE)
        env=Monitor(core,filename=str(log_dir/'monitor.csv'),allow_early_resets=True)
        if resume:
            agent=PPO.load(str(model_file),env=env,device='auto',
                           tensorboard_log=str(log_dir/'tensorboard'))
            print('REPRISE du modele, compteur initial:',agent.num_timesteps)
        else:
            agent=PPO('MultiInputPolicy',env,n_steps=128,batch_size=32,n_epochs=4,
                      learning_rate=3e-4,gamma=.99,gae_lambda=.95,ent_coef=.005,
                      verbose=1,device='auto',tensorboard_log=str(log_dir/'tensorboard'))
            print('NOUVEAU modele, compteur initial:',agent.num_timesteps)
        start_steps=int(agent.num_timesteps)
        cb=PauseDuringUpdates(core,prefix,meta)
        agent.learn(total_timesteps=int(TOTAL_TIMESTEPS),reset_num_timesteps=not resume,
                    callback=cb,tb_log_name=MODELE_NOM,progress_bar=False)
        core.safe_stop()
        meta=save_training(agent,prefix,meta)
        print('Modele:',model_file,'metadata:',meta_file)
        print('Actions ajoutees:',agent.num_timesteps-start_steps,'; compteur cumule:',agent.num_timesteps)
        print('Journaux de cette session:',log_dir)
        print('Version finale conservee :',archiver_agent(prefix,'fin_session'))
    except (KeyboardInterrupt,Exception) as exc:
        if core is not None:
            try:core.safe_stop()
            except Exception as stop_error:print('Erreur frein:',stop_error)
        if agent is not None:
            interrupted=PROJECT/'models'/'checkpoints'/(MODELE_NOM+'_interrupted_'+run_id)
            save_training(agent,interrupted,meta)
            print('Arret: sauvegarde separee:',interrupted.with_suffix('.zip'))
        raise
    finally:
        if core is not None: core.close()


## Executer la sequence

In [ ]:
# EXECUTER LA SEQUENCE — aucun reglage a modifier ici.
# On peut reexecuter uniquement cette cellule apres avoir reexecute REGLAGES,
# si toutes les definitions ont deja ete chargees dans le noyau.
executer_ouarti()


In [ ]:
# Diagnostic facultatif apres un premier essai : lire les journaux sans commander la voiture.
import json
from pathlib import Path
from collections import defaultdict

report_root=PROJECT/'reports'/('ppo_checkpoint_v3_stage'+str(ETAPE_APPRENTISSAGE))
# Chaque continuation a son dossier : pas de collision entre identifiants d'episodes.
candidates=list(report_root.glob('*/ppo_steps.jsonl'))
journal=max(candidates,key=lambda p:p.stat().st_mtime) if candidates else report_root/'ppo_steps.jsonl'
print('Journal analyse:',journal)
if not journal.exists():
    print('Aucun journal v3 pour le moment. Lancer volontairement un essai PPO court.')
else:
    episodes = defaultdict(list)
    with journal.open('r', encoding='utf8') as f:
        for line in f:
            try:
                rec = json.loads(line)
                episodes[rec['episode_id']].append(rec)
            except (ValueError, KeyError):
                continue
    print('10 derniers episodes :')
    print('ep | pas | reward | progression(m) | vitesse max(km/h) | gaz moy | frein moy | objectif | coupure PPO')
    for ep in sorted(episodes)[-10:]:
        steps = episodes[ep]
        n = len(steps)
        mean = lambda key: sum(float(p.get(key) or 0) for p in steps) / n
        print(f"{ep:>2} | {n:>3} | {sum(float(p.get('reward') or 0) for p in steps):>7.2f} | "
              f"{sum(float(p.get('progress_m') or 0) for p in steps):>14.2f} | "
              f"{max(float(p.get('speed_ms') or 0) for p in steps)*3.6:>18.1f} | "
              f"{mean('throttle_cmd'):>7.2f} | {mean('brake_cmd'):>9.2f} | "
              f"{any(bool(p.get('milestone_reached')) for p in steps)} | "
              f"{any(bool(p.get('rollout_cut')) for p in steps)}")
